# 02 — Preprocessing

1. Imputasi: **interpolasi linear** + fallback `ffill+bfill`.
2. **Causal smoothing** AOD (rolling mean, tanpa `center=True`).
3. **Split kronologis** 70/15/15.
4. `MinMaxScaler` **fit hanya pada train** untuk mencegah leakage.
5. Bentuk **sequences** dengan `lookback=30`.

output di `CAPSTONE/data/processed/<station>.csv` 

In [6]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from src import config as C
from src.data_loader import load_all_stations
from src.preprocessing import build_pipeline, impute_linear_then_fill, causal_smooth_aod

In [7]:
FEATURES = C.WEATHER_COLS + [C.AOD_COL, C.TARGET]
FEATURES

['temp', 'dew', 'humidity', 'precip', 'windspeed', 'AOD', 'ISPU PM2.5']

In [8]:
dfs = load_all_stations(reindex=True)

summaries = []
for name, df in dfs.items():
    df_imp = impute_linear_then_fill(df, FEATURES)
    n_miss_before = df[FEATURES].isna().sum().sum()
    n_miss_after = df_imp[FEATURES].isna().sum().sum()
    summaries.append({
        'station': name,
        'rows': len(df_imp),
        'missing_before': int(n_miss_before),
        'missing_after_impute': int(n_miss_after),
    })
    df_imp.to_csv(C.PROCESSED_DIR / f'{name}.csv', index=False)

summary = pd.DataFrame(summaries)
summary.to_csv(C.METRICS_DIR / '02_imputation_summary.csv', index=False)
summary

,station,rows,missing_before,missing_after_impute
0,us_embassy_1,1096,372,0
1,us_embassy_2,1096,545,0
2,jakarta_gbk,1096,1017,0
3,bundaran_hi,1096,492,0
4,kelapa_gading,1096,447,0
5,jagakarsa,1096,470,0
6,lubang_buaya,1096,479,0
7,kebun_jeruk,1096,493,0


## Demo build_pipeline untuk satu stasiun (DKI2 Kelapa Gading)

In [9]:
data = build_pipeline(dfs['kelapa_gading'], FEATURES, smooth_aod=False)
for k in ('X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test'):
    print(f'{k}: {data[k].shape}')

X_train: (737, 30, 7)
y_train: (737,)
X_val: (134, 30, 7)
y_val: (134,)
X_test: (135, 30, 7)
y_test: (135,)


In [10]:
# Bandingkan dengan smoothing aktif
data_sm = build_pipeline(dfs['kelapa_gading'], FEATURES, smooth_aod=True, smooth_window=7)
for k in ('X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test'):
    print(f'{k}: {data_sm[k].shape}')

X_train: (737, 30, 7)
y_train: (737,)
X_val: (134, 30, 7)
y_val: (134,)
X_test: (135, 30, 7)
y_test: (135,)
